# Multi-Agent Data Analyst: Evaluation Runner

Runs the real Groq/Gemini-backed agents against generated scenarios. Configure `.env` before executing.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.system_oneshot import run_oneshot_sql
from src.system_react import run_single_react_agent
from src.system_multi import run_multi_agent_team

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config.json').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config.json').exists():
    raise FileNotFoundError('Run this notebook from the project directory or its direct child.')
SCENARIOS_DIR = PROJECT_ROOT / 'scenarios'
LABELS_PATH = SCENARIOS_DIR / 'labels.json'


In [ ]:
with LABELS_PATH.open(encoding='utf-8') as labels_file:
    test_scenarios = json.load(labels_file)

results = []
for test in test_scenarios:
    scenario_id = test['scenario_id']
    db_path = SCENARIOS_DIR / f"scenario_{scenario_id:02d}.db"
    question = test['question']
    print(f"Evaluating scenario {scenario_id}: {test['planted_cause']}")

    row = {
        'scenario_id': scenario_id,
        'ground_truth_cause': test['planted_cause'],
        'ground_truth_entity': test['entity'],
        'ground_truth_dollar_impact': test['dollar_impact'],
    }
    try:
        oneshot = run_oneshot_sql(str(db_path), question)
        row.update({
            'oneshot_answer': oneshot['final_answer'],
            'oneshot_sql': oneshot['generated_sql'],
        })
    except Exception as exc:
        row['oneshot_error'] = f"{type(exc).__name__}: {exc}"

    try:
        react = run_single_react_agent(str(db_path), question)
        row.update({
            'react_answer': react['final_answer'],
            'react_steps': react['steps_taken'],
            'react_query_count': len(react['queries_executed']),
        })
    except Exception as exc:
        row['react_error'] = f"{type(exc).__name__}: {exc}"

    try:
        multi = run_multi_agent_team(str(db_path), question)
        row.update({
            'multi_agent_answer': multi['final_answer'],
            'multi_agent_audit_rounds': multi['audit_rounds'],
            'multi_agent_query_count': len(multi['queries_executed']),
        })
    except Exception as exc:
        row['multi_agent_error'] = f"{type(exc).__name__}: {exc}"

    results.append(row)
    print(f"Completed scenario {scenario_id}")


In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv(PROJECT_ROOT / 'evaluation_results.csv', index=False)
df_results
